## No memory

In [1]:
from langchain.agents import create_agent
from azure_openai_llm import create_azure_llm

from dotenv import load_dotenv
load_dotenv()

llm = create_azure_llm()

agent = create_agent(llm)

In [2]:
from langchain.messages import HumanMessage
from pprint import pprint

question = HumanMessage(content="Hello my name is Ali and my favourite colour is green")

response = agent.invoke(
    {"messages": [question]} 
)
pprint(response["messages"][-1].content)

("Hello Ali! It's nice to meet you. Green is a great color—very calming and "
 'refreshing. How can I assist you today?')


In [3]:
question = HumanMessage(content="What's my favourite colour?")

response = agent.invoke(
    {"messages": [question]} 
)

pprint(response["messages"][-1].content)

('I don’t know your favourite colour yet. If you tell me, I’d be happy to '
 'remember it for our conversation!')


## Memory

In [4]:
# langgraph checkpoint API.  Snapshot of the graph state at a given point in time. 
from langgraph.checkpoint.memory import InMemorySaver  


agent = create_agent(
    llm,
    checkpointer=InMemorySaver(),  
)

In [5]:
from langchain.messages import HumanMessage

question = HumanMessage(content="Hello my name is Ali and my favourite colour is green")
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [question]},
    config,  
)

In [6]:
pprint(response["messages"][-1].content)

("Hello Ali! It's nice to meet you. Green is a wonderful color—so refreshing "
 'and calming. How can I assist you today?')


In [7]:
question = HumanMessage(content="What's my favourite colour?")

response = agent.invoke(
    {"messages": [question]},
    config,  
)

pprint(response["messages"][-1].content)

'Your favourite colour is green.'


In [8]:
# Demonstrate storing custom data in memory for the same thread
user_data = {
    "name": "Ali",
    "favorite_color": "green",
    "city": "Lahore",
    "loyalty_tier": "gold"
}

# Save custom data into the thread memory
save_message = HumanMessage(
    content=(
        "Store this custom data in memory and use it for future answers:\n"
        f"{user_data}"
    )
)
agent.invoke({"messages": [save_message]}, config)

# Ask a follow-up question that requires the stored custom data
question = HumanMessage(
    content="From my saved profile, what city do I live in and what is my loyalty tier?"
)
response = agent.invoke({"messages": [question]}, config)
pprint(response["messages"][-1].content)

# Optional: inspect recent messages stored in memory for this thread
state = agent.get_state(config)
pprint(state.values["messages"][-4:])

('According to your saved profile, you live in Lahore and your loyalty tier is '
 'gold. How can I assist you further, Ali?')
[HumanMessage(content="Store this custom data in memory and use it for future answers:\n{'name': 'Ali', 'favorite_color': 'green', 'city': 'Lahore', 'loyalty_tier': 'gold'}", additional_kwargs={}, response_metadata={}, id='3e603668-dd2c-4de1-baa5-19bee4bec548'),
 AIMessage(content="Got it, Ali! I've stored your information:\n\n- Name: Ali  \n- Favorite color: green  \n- City: Lahore  \n- Loyalty tier: gold  \n\nI'll use this information to personalize my responses for you in the future. How can I assist you today?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 116, 'total_tokens': 170, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model

In [9]:
# Additional memory scenarios with the existing `agent` and `HumanMessage`

def ask(thread_config, text):
    result = agent.invoke({"messages": [HumanMessage(content=text)]}, thread_config)
    return result["messages"][-1].content

# Scenario 1: Thread isolation (different users/sessions)
config_a = {"configurable": {"thread_id": "memory-demo-A"}}
config_b = {"configurable": {"thread_id": "memory-demo-B"}}

ask(config_a, "My preferred database is PostgreSQL.")
ask(config_b, "My preferred database is MongoDB.")

print("Scenario 1A:", ask(config_a, "What database do I prefer?"))
print("Scenario 1B:", ask(config_b, "What database do I prefer?"))

# Scenario 2: Preference update within the same thread
config_update = {"configurable": {"thread_id": "memory-demo-update"}}

ask(config_update, "My deployment region is East US.")
ask(config_update, "Actually, update that: my deployment region is West Europe.")
print("Scenario 2:", ask(config_update, "Which deployment region should you use now?"))

# Scenario 3: Multi-step task memory
config_task = {"configurable": {"thread_id": "memory-demo-task"}}

ask(config_task, "I'm building a chatbot MVP. Deadline is Friday. Budget is $2,000.")
print("Scenario 3A:", ask(config_task, "Summarize my constraints in one sentence."))
print("Scenario 3B:", ask(config_task, "Given those constraints, suggest 3 practical next steps."))

# Optional: inspect latest stored messages for one scenario
task_state = agent.get_state(config_task)
print("\nRecent messages in Scenario 3 thread:")
pprint(task_state.values["messages"][-4:])

Scenario 1A: You prefer PostgreSQL as your database.
Scenario 1B: You prefer MongoDB.
Scenario 2: The deployment region to use now is **West Europe**.
Scenario 3A: You need to build a chatbot MVP with key features completed by Friday, within a budget of $2,000.
Scenario 3B: Here are three practical next steps given your Friday deadline and $2,000 budget:

1. **Choose a No-Code/Low-Code Chatbot Platform:** Select an easy-to-use platform like Landbot, Chatfuel, or Tars that lets you quickly build and deploy a chatbot without heavy development effort.

2. **Define Core Use Cases and Script:** Prioritize and clearly outline the essential interactions your chatbot must handle to keep the scope manageable and meet the deadline.

3. **Set Up and Test Rapidly:** Build the chatbot on the chosen platform, then test it internally to iron out major issues before launch to ensure it meets user needs within your timeline and budget.

Recent messages in Scenario 3 thread:
[HumanMessage(content='Summa